# Phase 3c -- diagnostics micro-session (A100-40GB, High-RAM **OFF**)

Four sanity/bisection questions left open by Phase 3b, bundled into one
from-scratch session so setup is paid once (9 cells, 4 server launches,
~1.5-2h, ~20-25 units). Generator + interpretation grid:
`configs/diagnostics/generate_diagnostics.py`.

1. **tau-eager-short** -- is the probe's tau=1.14 a long-context finding or an
   eager artifact? (tau~2.85 here = finding is real)
2. **slong-eager-base** -- no-spec eager baseline at 7.4k context: measures
   S-main at long context within the eager regime (probe refs: 34.0 tok/s @ c1,
   166.4 @ c8).
3. **cudagraph-probe** -- compile ON, CUDA graphs OFF on the crashing corner.
   **A crash here is an informative outcome, not a failure of the session** --
   do not retry it; it feeds the upstream vLLM issue only.
4. **backendpin-flashinfer** -- FP16-KV c8 with FLASHINFER pinned: bounds the
   backend component inside every K contrast (k_stress ref: ~221 tok/s on
   FLASH_ATTN).

Push the repo before starting -- Colab pulls from GitHub.

In [ ]:
# 1. Verify the 40GB card (High-RAM OFF)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import subprocess
gpu = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert "A100" in gpu, "Not an A100 - reconnect until Colab honors the selection."
mem = int(subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()[0])
assert mem < 50000, ("Got the 80GB A100 (%d MiB). Toggle High-RAM OFF and reconnect: "
                     "these diagnostics are compared against Phase-3b 40GB cells.") % mem
units_before = input("Compute-units balance right now: ")

In [ ]:
# 2. Repo + Spec-Bench (long documents are built from its RAG passages)
%cd /content
!git clone https://github.com/manojarulmurugan/SpecDecoding-Study-vLLM-SGLang.git repo 2>/dev/null || (cd repo && git pull)
%cd /content/repo
!git clone --depth 1 https://github.com/hemingkx/Spec-Bench.git external/Spec-Bench 2>/dev/null || echo "Spec-Bench already present"
!test -f external/Spec-Bench/data/spec_bench/question.jsonl && echo "RAG data OK"

In [ ]:
# 3. Isolated vLLM environment (PREREQ_RESULTS Check 6 recipe; ~6-8 min)
%pip install -q virtualenv
!virtualenv -q /content/vllm_env
!/content/vllm_env/bin/pip install -q vllm==0.24.0 datasets pyyaml requests pytest ninja
!/content/vllm_env/bin/python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 4. HF auth
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("token set:", bool(os.environ["HF_TOKEN"]))

In [ ]:
# 4b. Pre-download checkpoints (bounded, retried; automatic browser-UA curl
# fallback if HF's hf-client CDN route is broken -- PREREQ 2026-07-14/15).
# If hf is KNOWN-broken right now, use:  ... predownload.py --curl-only
!/content/vllm_env/bin/python scripts/predownload.py

In [ ]:
#4b Alt: restore the HF cache from the Phase 3b Drive backup instead
# (specdecoding_hf_cache -- the actual backup dir from the Phase 3b session,
# not a guessed path). Covers both checkpoints this phase needs.
from google.colab import drive
drive.mount('/content/drive')

backup_dir = "/content/drive/MyDrive/specdecoding_hf_cache"
!mkdir -p /root/.cache/huggingface
!rsync -a --info=progress2 {backup_dir}/ /root/.cache/huggingface/
print("Restored. Verify predownload sees everything cached:")
!/content/vllm_env/bin/python scripts/predownload.py


In [ ]:
# 5. Harness self-check (~1 min, no GPU)
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m pytest tests -q

In [ ]:
# 6. Sanity: 9 cells in 4 server groups; check the four distinct commands
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/diagnostics/diag_*.yaml" --results-dir results --dry-run 2>&1 | grep -E "server command|run\(s\)"

In [ ]:
# 7. THE SWEEP. cudagraph-probe is ordered LAST: if it crashes the server,
# that IS its result (status=failed = bug pinned to compiled kernels beyond
# CUDA graphs; status=ok = bug is specifically CUDA-graph capture). Do NOT
# re-run it on a crash.
!PATH="/content/vllm_env/bin:$PATH" /content/vllm_env/bin/python -m harness.sweep "configs/diagnostics/diag_tau-eager-short_*.yaml" "configs/diagnostics/diag_slong-eager-base_*.yaml" "configs/diagnostics/diag_backendpin-flashinfer_*.yaml" "configs/diagnostics/diag_cudagraph-probe_*.yaml" --results-dir results

In [ ]:
# 8. Verdicts (references from verified Phase-2/3/3b records)
import json, glob
recs = {json.load(open(f))["run_id"]: json.load(open(f)) for f in glob.glob("results/runs/diag_*.json")}
def m(rid, key): return (recs.get(rid, {}).get("measured") or {}).get(key)
def status(rid): return recs.get(rid, {}).get("status", "MISSING")

print("=== 1. tau check (refs: factorial short-ctx compiled tau~2.85; probe long-ctx eager tau~1.14)")
taus = [m("diag_tau-eager-short_c1_r%d" % r, "accepted_length_tau") for r in (0, 1)]
print("  eager short-ctx tau:", taus)
if all(t and t > 2.4 for t in taus):
    print("  VERDICT: eager is innocent -> tau collapse is a REAL long-context finding")
elif all(t and t < 1.6 for t in taus):
    print("  VERDICT: tau collapses even at short context under eager -> EAGER ARTIFACT; probe tau story needs rework")
else:
    print("  VERDICT: ambiguous / partial -- inspect records before concluding")

print()
print("=== 2. S-main at long context, within eager regime (probe refs: 34.0 tok/s @ c1, 166.4 @ c8)")
for conc, ref in ((1, 34.0), (8, 166.4)):
    g = [m("diag_slong-eager-base_c%d_r%d" % (conc, r), "goodput_tok_s") for r in (0, 1)]
    gm = sum(x for x in g if x) / max(1, len([x for x in g if x]))
    print("  c%d: no-spec baseline %.1f tok/s -> S factor x%.2f %s"
          % (conc, gm, ref / gm if gm else float("nan"),
             "(S is COUNTERPRODUCTIVE at long context)" if gm and ref / gm < 1 else ""))

print()
print("=== 3. cudagraph bisection")
s = status("diag_cudagraph-probe_c1_r0")
print("  status:", s)
print("  ok     -> bug is specifically CUDA-GRAPH CAPTURE (upstream report: cudagraph_mode=FULL_AND_PIECEWISE crashes, NONE survives)")
print("  failed -> bug is in the compiled eagle_head kernels beyond graph capture")

print()
print("=== 4. backend pin (ref: k_stress fp16kv c8 on FLASH_ATTN ~221 tok/s)")
g = [m("diag_backendpin-flashinfer_c8_r%d" % r, "goodput_tok_s") for r in (0, 1)]
print("  FLASHINFER fp16-KV c8:", g, "-> backend component of the K contrast = this vs 221")

In [ ]:
# 9. Preserve everything
units_after = input("Compute-units balance now: ")
print("Units burned:", units_before, "->", units_after)
!zip -qr phase3c_diagnostics_results.zip results
from google.colab import files
files.download("phase3c_diagnostics_results.zip")